# D4-2: FLOP-priced scoring on a higher-dimensional system (validation-only; the test family is never generated)

Follow-up to D4-0b (`docs/research-notes/2026-09-29-d4-0b-where-adaptivity-pays.md`). The full design, the ledger formulas, the decision rules and the list of changes made before this run are in `docs/plans/d4-2-plan.md`; read that first. **This notebook is a design study, not a confirmatory gate.** It uses a scorer-training family (seed 4004), a tuning family (seed 3003) and a validation family (seed 1001). **The test family (seed 2002) is never generated or read** (`highdim.FAMILY_SEEDS` does not contain it, and `cell_instances` refuses it). It runs on CPU.

**What it does.** For seven cells (a base cell and six one-factor variations: dimension `m` = 4 and 64, wide pulses, amplitude scale 1, depth 7 and 11) it compares, at equal compute:

- `uniform_pass`, no scoring;
- the D4-0b step-doubling arms `residual`, `goal_local`, `adjoint` (scoring charged as `2n`, `2n`, `3n` CN steps per pass);
- cheap, solve-free arms priced in floating-point operations and converted to CN steps: `indicator` (forcing quadrature discrepancy; a diagnostic that cannot see the homogeneous error), `cheap` (that plus a third divided difference of the numerical solution) and `cheap_adjoint` (the same weighted by a tabulated continuous co-state, whose one-off setup is reported separately);
- `amortised`: a small MLP trained on the train family to predict the exact co-state-weighted local error from nine cheap features (no co-state input), priced by the FLOPs of its features and network; its training cost is reported separately;
- the hypothetical ceiling `adjoint_free`: the step-doubling `adjoint` arm with scoring priced at zero, read off the same traces.

All arms mark with Dörfler's rule; `θ` is chosen per (cell, arm) on the tuning family from `{0.5, 0.7, 0.85, 0.95, 0.98, 0.995}`, and the network size from `{8, 16, 32}` on the tuning family at `θ = 0.95`. Compute to reach 10 %, 3 % and 1 % of the initial-to-finest gap is priced by **log-log interpolation between passes for every arm** (primary, fixed in advance); semilog interpolation and the plain staircase are sensitivity analyses computed from the same traces.

**Frozen decision rules** (from the plan, §8; a CI "below 0" is for `mean(log(compute_a / compute_b))` at two adjacent targets):

- **R1, an amortised scorer pays:** `amortised` beats `uniform_pass` at its FLOP price.
- **R3, the co-state weight helps a cheap estimator:** `cheap_adjoint` beats `cheap`.
- **D4-1 candidate cell:** R1 and R3 both hold.
- Reported, not gated: the retained headroom `(log u − log a) / (log u − log f)`, the price ratios, the one-off costs and break-even instance counts, and the comparison of `amortised` with the cheap arms.

**Limits.** The goal `c` is fixed per system, so the co-state weight is a fixed function of time and the network can absorb it: this is an amortised adjoint scorer (supervised by the adjoint-weighted error), not a direct critic, and nothing here is about H2. FLOPs are counted from the code path, not measured; wall-clock timings are printed as a sanity check only. The generation of exact labels for training (reference solutions) is not charged. The instances are generated from frozen seeds and are **not real data**.

In [ ]:
# ================================================================
# 0. Package (CPU is enough)
# ================================================================
import os
import subprocess
import sys
from pathlib import Path

IN_COLAB = 'google.colab' in sys.modules
REPO_URL = 'https://github.com/romusking/para_001'
REPO_REF = 'main'  # branch, tag or commit SHA
REPO_DIR = '/content/para_001' if IN_COLAB else str(Path.cwd().resolve().parents[1])

if IN_COLAB:
    if not os.path.exists(REPO_DIR):
        subprocess.run(['git', 'clone', '--quiet', REPO_URL, REPO_DIR], check=True)
    subprocess.run(['git', '-C', REPO_DIR, 'fetch', '--quiet', '--all'], check=False)
    for candidate in (f'origin/{REPO_REF}', REPO_REF):
        resolved = subprocess.run(['git', '-C', REPO_DIR, 'rev-parse', '--verify', '--quiet', candidate + '^{commit}'],
                                  capture_output=True, text=True)
        if resolved.returncode == 0:
            subprocess.run(['git', '-C', REPO_DIR, 'checkout', '--quiet', '--detach', resolved.stdout.strip()], check=True)
            break
    else:
        raise RuntimeError(f'Cannot resolve {REPO_REF!r}.')
REPO_COMMIT = subprocess.check_output(['git', '-C', REPO_DIR, 'rev-parse', 'HEAD'], text=True).strip()
REPO_DIRTY = bool(subprocess.check_output(['git', '-C', REPO_DIR, 'status', '--porcelain'], text=True).strip())
sys.path.insert(0, os.path.join(REPO_DIR, 'src'))
try:
    from adjointrwm.domains import highdim  # noqa: F401
except ImportError as error:
    raise RuntimeError(f'{REPO_REF!r} predates the D4-2 module; set REPO_REF to a commit that has it.') from error

if IN_COLAB:
    from google.colab import drive

    drive.mount('/content/drive', force_remount=False)
    RUN_ROOT = Path('/content/drive/MyDrive/Colab Notebooks/AdjointRWM_Production/runs')
else:
    RUN_ROOT = Path.home() / 'adjointrwm_runs'
print('adjointrwm at', REPO_COMMIT, '(uncommitted changes!)' if REPO_DIRTY else '', '| runs ->', RUN_ROOT)

In [ ]:
# ================================================================
# 1. Frozen configuration, cells, instances and the immutable run directory
# ================================================================
import dataclasses
import datetime
import json
import platform
import time
from dataclasses import asdict, dataclass

import numpy as np
import pandas as pd

from adjointrwm.domains import AdaptiveTimeSteppingDomain, highdim as hd
from adjointrwm.io import atomic_write_json, atomic_write_text, sha256_json


@dataclass(frozen=True)
class D42Config:
    cells: tuple = tuple(hd.CELLS)                 # base, m4, m64, wide, amp1, depth7, depth11
    train_instances: int = 30                      # scorer training family (seed 4004)
    tuning_instances: int = 10                     # theta and network size (seed 3003)
    validation_instances: int = 20                 # reported (seed 1001)
    # The test family (seed 2002) is deliberately never generated in this notebook.
    thetas: tuple = (0.5, 0.7, 0.85, 0.95, 0.98, 0.995)
    step_doubling_arms: tuple = ('residual', 'goal_local', 'adjoint')
    cheap_arms: tuple = ('indicator', 'cheap', 'cheap_adjoint')
    hidden_sizes: tuple = (8, 16, 32)
    hidden_selection_theta: float = 0.95
    scorer_training_theta: float = 0.85            # marking fraction of the training-mesh policies
    scorer_max_passes: int = 40
    scorer_per_state: int = 256
    scorer_max_samples: int = 200000
    scorer_seed: int = 0
    gap_fractions: tuple = (0.1, 0.03, 0.01)
    cap_in_finest_grids: float = 2.0
    max_passes: int = 40
    decision_scales: tuple = (1.0, 0.0)            # 1 = real price; 0 = hypothetical free scoring (read only for the adjoint ceiling)
    primary_method: str = 'loglog'
    methods: tuple = ('loglog', 'semilog', 'staircase')
    bootstrap_resamples: int = 10000


CFG = D42Config()
CONFIG_DICT = asdict(CFG)
CELL_TABLE = {name: asdict(hd.CELLS[name]) for name in CFG.cells}
CONFIG_HASH = sha256_json({'config': CONFIG_DICT, 'cells': CELL_TABLE, 'family_seeds': hd.FAMILY_SEEDS})
RUN_ID = 'd4_2_flop_scoring_' + datetime.datetime.now(datetime.UTC).strftime('%Y%m%dT%H%M%SZ')
RUN_DIR = RUN_ROOT / RUN_ID
if RUN_DIR.exists():
    raise RuntimeError(f'{RUN_DIR} exists; run directories are immutable.')
PATHS = {k: RUN_DIR / k for k in ('config', 'artifacts', 'artifacts/scorers', 'figures', 'reports', 'tests')}
for path in PATHS.values():
    path.mkdir(parents=True, exist_ok=True)

DOMAIN = AdaptiveTimeSteppingDomain(objective_kind='bound')
COUNTS = {'train': CFG.train_instances, 'tuning': CFG.tuning_instances, 'validation': CFG.validation_instances}
INSTANCES = {(cell, family): hd.cell_instances(cell, family, count) for cell in CFG.cells for family, count in COUNTS.items()}
# Provenance without per-instance JSON: instances are regenerated from frozen seeds by hd.cell_instances (tested), their full parameters
# are stored in parquet (binary, small), and hashes of the generated objects let anyone check a regeneration.
SYSTEMS = {cell: {'m': hd.CELLS[cell].m, 'A': INSTANCES[(cell, 'tuning')][0].A, 'goal': INSTANCES[(cell, 'tuning')][0].goal} for cell in CFG.cells}
MANIFEST = {cell: {'system_seed': hd.CELLS[cell].system_seed, 'system_sha256': sha256_json(SYSTEMS[cell]),
                   **{family: {'seed': hd.FAMILY_SEEDS[family], 'count': count, 'names': [INSTANCES[(cell, family)][0].name, INSTANCES[(cell, family)][-1].name],
                               'sha256': sha256_json([asdict(i) for i in INSTANCES[(cell, family)]])} for family, count in COUNTS.items()}}
            for cell in CFG.cells}
pd.DataFrame([{'cell': cell, 'm': SYSTEMS[cell]['m'], 'A_flat': [x for row in SYSTEMS[cell]['A'] for x in row], 'goal': list(SYSTEMS[cell]['goal'])}
              for cell in CFG.cells]).to_parquet(PATHS['config'] / 'systems.parquet', index=False)
pd.DataFrame([{'cell': cell, 'family': family, 'name': i.name, 'y0': list(i.y0), 'n_fine': i.n_fine,
               'pulses_json': json.dumps([asdict(p) for p in i.pulses], separators=(',', ':'))}
              for (cell, family), insts in INSTANCES.items() for i in insts]).to_parquet(PATHS['config'] / 'instances.parquet', index=False)
atomic_write_json(PATHS['config'] / 'run_config.json', {
    'run_id': RUN_ID, 'config': CONFIG_DICT, 'cells': CELL_TABLE, 'family_seeds': hd.FAMILY_SEEDS, 'config_sha256': CONFIG_HASH,
    'repo_commit': REPO_COMMIT, 'repo_dirty': REPO_DIRTY, 'notebook': 'notebooks/04-domains/d4_2_flop_priced_scoring.ipynb',
    'plan': 'docs/plans/d4-2-plan.md', 'domain_spec': DOMAIN.spec.to_dict(), 'domain_spec_sha256': DOMAIN.spec.checksum(),
    'test_family_read': False,
})
atomic_write_json(PATHS['config'] / 'instances_manifest.json', {**MANIFEST, 'sha256': sha256_json(MANIFEST)})
atomic_write_json(PATHS['config'] / 'environment.json', {
    'python': sys.version, 'platform': platform.platform(), 'numpy': np.__version__, 'pandas': pd.__version__,
    'hardware': 'CPU (FLOPs are counted, not measured; timings are not a systems claim)',
})
print('Run:', RUN_ID, '| config', CONFIG_HASH[:12], '|', len(CFG.cells), 'cells')

In [ ]:
# ================================================================
# 2. Correctness and ledger checks (before anything else is read)
# ================================================================
from adjointrwm.domains import LinearODEInstance
from adjointrwm.domains.linear_ode import ODEState, reference_solution

rows = []
for cell in CFG.cells:
    inst = INSTANCES[(cell, 'tuning')][0]
    m, pulses = inst.m, len(inst.pulses)
    # (a) the step cost is the operations of CrankNicolson.steps, counted from its code path
    counted_step = 2 * (2 * m * m) + m + m + m + 2 * pulses * (2 * m + 8)
    # (b) the tabulated co-state approaches the discrete co-state on a fine uniform mesh (error of the continuous proxy)
    level_step = 2 ** 1
    nodes = tuple(range(0, inst.n_fine + 1, level_step))
    state = ODEState(inst, nodes, DOMAIN.stepper(inst).solve(nodes))
    discrete = DOMAIN.costate(state)
    proxy_gap = float(np.linalg.norm(discrete - hd.costate_table(inst)[np.asarray(nodes[1:])]) / np.linalg.norm(discrete))
    # (c) the forcing indicator vanishes without forcing; (d) batch application equals sequential application
    silent = dataclasses.replace(inst, pulses=())
    silent_state = DOMAIN.initial_state(silent)
    indicator_zero = bool(np.abs(hd.forcing_discrepancy(silent, silent_state)).max() == 0.0)
    start = DOMAIN.initial_state(inst)
    picked = [DOMAIN.legal_candidates(start)[j] for j in (1, 5, 6)]
    batch = DOMAIN.apply_batch(start, picked)
    sequential = start
    for candidate in picked:
        sequential = DOMAIN.apply(sequential, candidate)
    weighted = DOMAIN.weighted_local_errors(batch, inst).sum()
    # (e) the reference solution is converged: one more level of the finest grid changes y(T) negligibly
    finer = LinearODEInstance(**{**inst.__dict__, 'max_depth': inst.max_depth + 1})
    rows.append({'cell': cell, 'm': m, 'flops_step_formula': hd.flops_step(inst), 'flops_step_counted': counted_step,
                 'costate_proxy_rel_gap_fine_mesh': proxy_gap, 'indicator_zero_without_forcing': indicator_zero,
                 'batch_equals_sequential_nodes': batch.nodes == sequential.nodes,
                 'batch_vs_sequential_max_abs_diff': float(np.abs(batch.y - sequential.y).max()),
                 'representation_abs_diff': float(abs(weighted - DOMAIN.signed_error(batch, inst))),
                 'abs_qoi_error': abs(DOMAIN.signed_error(batch, inst)),
                 'reference_max_abs_diff_yT': float(np.abs(reference_solution(inst)[-1] - reference_solution(finer)[-1]).max()),
                 'features_finite': bool(np.isfinite(hd.scorer_features(inst, start)).all())})
checks = pd.DataFrame(rows)
# wall-clock sanity check (NOT a claim): one CN step for every interval versus features + network for every interval
PROBE_CELL = max(CFG.cells, key=lambda c: hd.CELLS[c].m)      # the highest-dimensional cell
inst = INSTANCES[(PROBE_CELL, 'tuning')][0]
nodes = tuple(range(0, inst.n_fine + 1, 2 ** max(inst.max_depth - 6, 0)))
state = ODEState(inst, nodes, DOMAIN.stepper(inst).solve(nodes))
probe_model = hd.train_scorer(np.random.default_rng(0).normal(size=(500, hd.FEATURE_DIM)), np.random.default_rng(1).normal(size=500), np.repeat(np.arange(10), 50), max_epochs=2)
stepper, t = DOMAIN.stepper(inst), inst.time_of(state.nodes)
started = time.perf_counter()
for _ in range(20):
    stepper.steps(state.y[:-1], t[:-1], np.diff(t))
step_time = (time.perf_counter() - started) / 20
started = time.perf_counter()
for _ in range(20):
    probe_model.predict_log10(hd.scorer_features(inst, state))
scorer_time = (time.perf_counter() - started) / 20
CORRECTNESS = {
    'flop_formula_matches_counted_code_path': bool((checks['flops_step_formula'] == checks['flops_step_counted']).all()),
    'costate_proxy_close_on_fine_mesh': bool((checks['costate_proxy_rel_gap_fine_mesh'] < 0.05).all()),
    'indicator_zero_without_forcing': bool(checks['indicator_zero_without_forcing'].all()),
    'batch_equals_sequential': bool(checks['batch_equals_sequential_nodes'].all() and (checks['batch_vs_sequential_max_abs_diff'] < 1e-10).all()),
    'error_representation_on_batch_grids': bool((checks['representation_abs_diff'] <= 1e-9 * np.maximum(1.0, checks['abs_qoi_error'])).all()),
    'reference_converged': bool((checks['reference_max_abs_diff_yT'] < 1e-8).all()),
    'features_finite': bool(checks['features_finite'].all()),
    'max_costate_proxy_rel_gap': float(checks['costate_proxy_rel_gap_fine_mesh'].max()),
    'max_representation_abs_diff': float(checks['representation_abs_diff'].max()),
    'max_reference_diff': float(checks['reference_max_abs_diff_yT'].max()),
    'wall_clock_sanity_seconds': {'cell': PROBE_CELL, 'intervals': len(state.nodes) - 1, 'one_step_all_intervals': step_time,
                                  'features_and_network_all_intervals': scorer_time, 'note': 'CPU sandbox timings; not a claim'},
}
CORRECTNESS['passed'] = bool(all(v for k, v in CORRECTNESS.items() if isinstance(v, bool)))
atomic_write_json(PATHS['tests'] / 'correctness.json', CORRECTNESS)
checks.to_csv(PATHS['tests'] / 'ledger_and_batch_checks.csv', index=False)
print(json.dumps(CORRECTNESS, indent=2))
if not CORRECTNESS['passed']:
    raise RuntimeError('Correctness failed: nothing below may be read. See tests/.')

In [ ]:
# ================================================================
# 3. Train the scorer, choose its size and theta on the train and tuning families only
# ================================================================
from adjointrwm.domains import marking_policy, uniform_pass_policy

STEP_DOUBLING = {kind: (lambda th, kind=kind: marking_policy(kind, th)) for kind in CFG.step_doubling_arms}
CHEAP = {'indicator': hd.indicator_policy, 'cheap': hd.cheap_policy, 'cheap_adjoint': hd.cheap_adjoint_policy}
ARMS = list(CFG.step_doubling_arms) + list(CFG.cheap_arms) + ['amortised']
SCORERS, HIDDEN_ROWS, TUNING_ROWS, SELECTED_THETA, SELECTED_HIDDEN, TRAIN_STEPS = {}, [], [], {}, {}, {}
started = time.perf_counter()
for cell in CFG.cells:
    train, tuning = INSTANCES[(cell, 'train')], INSTANCES[(cell, 'tuning')]
    x, y, groups = hd.collect_scorer_data(
        DOMAIN, train, [uniform_pass_policy(), marking_policy('residual', CFG.scorer_training_theta), marking_policy('adjoint', CFG.scorer_training_theta)],
        max_passes=CFG.scorer_max_passes, per_state=CFG.scorer_per_state, seed=CFG.scorer_seed)
    candidates = {}
    for h in CFG.hidden_sizes:
        model = hd.train_scorer(x, y, groups, seed=CFG.scorer_seed, hidden=(h, h), max_samples=CFG.scorer_max_samples)
        cost = hd.tune_theta(DOMAIN, tuning, lambda th, model=model: hd.amortised_policy(model, th), (CFG.hidden_selection_theta,),
                             CFG.gap_fractions, CFG.cap_in_finest_grids, CFG.max_passes, CFG.primary_method)[CFG.hidden_selection_theta]
        candidates[h] = (model, cost)
        HIDDEN_ROWS.append({'cell': cell, 'hidden': h, 'tuning_geometric_mean_compute': cost, 'val_rmse_log10': model.report['val_rmse_log10'],
                            'epochs_run': model.report['epochs_run'], 'training_flops': model.report['training_flops'], 'n_train': model.report['n_train']})
    best = min(candidates, key=lambda h: candidates[h][1])
    SCORERS[cell], SELECTED_HIDDEN[cell] = candidates[best][0], best
    atomic_write_json(PATHS['artifacts/scorers'] / f'{cell}.json', SCORERS[cell].to_json())
    TRAIN_STEPS[cell] = SCORERS[cell].report['training_flops'] / hd.flops_step(tuning[0])
    for arm in ARMS:
        make = (lambda th: hd.amortised_policy(SCORERS[cell], th)) if arm == 'amortised' else STEP_DOUBLING.get(arm) or CHEAP[arm]
        options = hd.tune_theta(DOMAIN, tuning, make, CFG.thetas, CFG.gap_fractions, CFG.cap_in_finest_grids, CFG.max_passes, CFG.primary_method)
        SELECTED_THETA[(cell, arm)] = min(options, key=options.get)
        TUNING_ROWS.append({'cell': cell, 'arm': arm, **{f'theta_{t:g}': options[t] for t in CFG.thetas}, 'selected': SELECTED_THETA[(cell, arm)],
                            'selected_at_grid_edge': SELECTED_THETA[(cell, arm)] in (min(CFG.thetas), max(CFG.thetas))})
    print(f'{cell}: hidden {best}, scorer val rmse {SCORERS[cell].report["val_rmse_log10"]:.2f} log10, {time.perf_counter() - started:.0f}s elapsed')
HIDDEN = pd.DataFrame(HIDDEN_ROWS)
TUNING = pd.DataFrame(TUNING_ROWS)
HIDDEN.to_csv(PATHS['artifacts'] / 'hidden_size_selection.csv', index=False)
TUNING.to_csv(PATHS['artifacts'] / 'theta_tuning.csv', index=False)
print(TUNING[['cell', 'arm', 'selected', 'selected_at_grid_edge']].to_string(index=False))

# Freeze everything that validation is about to be judged against, before the validation family is generated or read.
FROZEN = {'config_sha256': CONFIG_HASH, 'selected_theta': {f'{c}/{a}': v for (c, a), v in SELECTED_THETA.items()},
          'selected_hidden': SELECTED_HIDDEN, 'scorer_sha256': {c: sha256_json(SCORERS[c].to_json()) for c in CFG.cells},
          'rules': {'R1': 'amortised beats uniform_pass at two adjacent targets (primary method log-log)',
                    'R3': 'cheap_adjoint beats cheap at two adjacent targets',
                    'candidate_for_d4_1': 'R1 and R3', 'beats': 'CI of mean log compute ratio below 0'},
          'primary_method': CFG.primary_method, 'written_utc': datetime.datetime.now(datetime.UTC).isoformat()}
FROZEN['sha256'] = sha256_json({k: v for k, v in FROZEN.items() if k != 'written_utc'})
atomic_write_json(PATHS['reports'] / 'frozen_before_validation.json', FROZEN)
print('frozen', FROZEN['sha256'][:12])

In [ ]:
# ================================================================
# 4. Validation family: compute to reach each target at the real price, and the hypothetical free ceiling
# ================================================================
import dataclasses

from adjointrwm.domains import evaluate_work_precision, work_precision_summary

POLICIES = {}
for cell in CFG.cells:
    arms = [uniform_pass_policy()]
    for arm in ARMS:
        theta = SELECTED_THETA[(cell, arm)]
        make = (lambda th: hd.amortised_policy(SCORERS[cell], th)) if arm == 'amortised' else STEP_DOUBLING.get(arm) or CHEAP[arm]
        arms.append(dataclasses.replace(make(theta), name=arm))
    POLICIES[cell] = arms
PAIRS = [('amortised', 'uniform_pass'), ('cheap', 'uniform_pass'), ('cheap_adjoint', 'uniform_pass'), ('adjoint', 'uniform_pass'),
         ('residual', 'uniform_pass'), ('goal_local', 'uniform_pass'), ('indicator', 'uniform_pass'),
         ('cheap_adjoint', 'cheap'), ('amortised', 'cheap'), ('amortised', 'cheap_adjoint'), ('adjoint_free', 'amortised'),
         ('adjoint_free', 'uniform_pass'), ('adjoint', 'amortised')]
TARGETS = list(hd.TARGETS)


def cap_of(instance):
    return CFG.cap_in_finest_grids * instance.n_fine


started = time.perf_counter()
FRAMES, SUMMARY = {}, {}
for cell in CFG.cells:
    parts = [evaluate_work_precision(DOMAIN, [inst], POLICIES[cell], lambda d, i: hd.gap_targets(d, i, CFG.gap_fractions), compute_cap=cap_of(inst),
                                     max_steps=CFG.max_passes, random_draws=1, decision_scales=CFG.decision_scales, methods=CFG.methods)
             for inst in INSTANCES[(cell, 'validation')]]
    frame = pd.concat(parts, ignore_index=True)
    frame.insert(0, 'cell', cell)
    FRAMES[cell] = frame
    cap = float(cap_of(INSTANCES[(cell, 'validation')][0]))     # identical for every instance of a cell
    SUMMARY[cell] = {}
    for method in CFG.methods:
        sub = hd.with_free_reference(frame[frame['method'] == method].drop(columns=['cell', 'method']))
        SUMMARY[cell][method] = work_precision_summary(sub, PAIRS, compute_cap=cap, num_resamples=CFG.bootstrap_resamples)['1']
    print(f'{cell}: {time.perf_counter() - started:.0f}s elapsed')
ALL = pd.concat(FRAMES.values(), ignore_index=True)
ALL.to_parquet(PATHS['artifacts'] / 'validation_work_precision.parquet', index=False)
atomic_write_json(PATHS['artifacts'] / 'validation_summary.json', {
    'selected_theta': FROZEN['selected_theta'], 'selected_hidden': SELECTED_HIDDEN, 'pairs': PAIRS, 'methods': list(CFG.methods),
    'primary_method': CFG.primary_method, 'summary': SUMMARY,
    'note': 'adjoint_free is the adjoint arm with scoring priced at zero (hypothetical); every other arm is at its real price'})
for cell in CFG.cells:
    for target in TARGETS:
        e = SUMMARY[cell][CFG.primary_method][target]
        print(f"{cell:8s} {target:6s} " + ' '.join(f"{p}={e['median_compute'][p]:.0f}" for p in ('uniform_pass', 'amortised', 'cheap', 'cheap_adjoint', 'adjoint', 'adjoint_free')))

In [ ]:
# ================================================================
# 5. Frozen rules, one-off costs, price ratios, table, figure and report
# ================================================================
CELL_ROWS, PRICES, BREAK_EVEN, HEADROOM = [], {}, {}, {}
for cell in CFG.cells:
    inst = INSTANCES[(cell, 'validation')][0]
    dims = SCORERS[cell].dims
    PRICES[cell] = hd.price_report(inst, dims)
    per_method = {m: hd.cell_rules(SUMMARY[cell][m]) for m in CFG.methods}
    primary = per_method[CFG.primary_method]
    cap = float(cap_of(inst))
    frame = hd.with_free_reference(FRAMES[cell][FRAMES[cell]['method'] == CFG.primary_method].drop(columns=['cell', 'method']))
    HEADROOM[cell] = hd.retained_headroom(frame, cap)
    tight = TARGETS[-1]
    uni = frame[(frame['policy'] == 'uniform_pass') & (frame['target'] == tight)].set_index('instance')['compute'].clip(upper=cap)

    def arm_compute(arm):
        return frame[(frame['policy'] == arm) & (frame['target'] == tight)].set_index('instance')['compute'].clip(upper=cap).reindex(uni.index)

    BREAK_EVEN[cell] = {
        'amortised_training_steps': TRAIN_STEPS[cell], 'amortised_break_even_instances': hd.break_even_instances(TRAIN_STEPS[cell], uni, arm_compute('amortised')),
        'costate_table_setup_steps': PRICES[cell]['table_setup_steps'], 'cheap_adjoint_break_even_instances': hd.break_even_instances(PRICES[cell]['table_setup_steps'], uni, arm_compute('cheap_adjoint'))}
    row = {'cell': cell, 'm': inst.m, 'hidden': SELECTED_HIDDEN[cell], 'R1': primary['R1_amortised_beats_uniform'], 'R3': primary['R3_costate_weight_helps_cheap_estimator'],
           'candidate_regime_for_d4_1': primary['candidate_regime_for_d4_1'],
           'R1_semilog': per_method['semilog']['R1_amortised_beats_uniform'], 'R1_staircase': per_method['staircase']['R1_amortised_beats_uniform'],
           'R3_semilog': per_method['semilog']['R3_costate_weight_helps_cheap_estimator'], 'R3_staircase': per_method['staircase']['R3_costate_weight_helps_cheap_estimator'],
           'amortised_price_ratio_to_step_doubling': PRICES[cell]['ratio_to_step_doubling']['amortised'],
           'cheap_price_ratio_to_step_doubling': PRICES[cell]['ratio_to_step_doubling']['cheap'],
           'cheap_adjoint_price_ratio_to_step_doubling': PRICES[cell]['ratio_to_step_doubling']['cheap_adjoint']}
    for t in TARGETS:
        d = SUMMARY[cell][CFG.primary_method][t]['differences']
        row.update({f'amortised_over_uniform_{t}': d['amortised / uniform_pass']['ratio_of_geometric_means'],
                    f'cheap_over_uniform_{t}': d['cheap / uniform_pass']['ratio_of_geometric_means'],
                    f'cheap_adjoint_over_uniform_{t}': d['cheap_adjoint / uniform_pass']['ratio_of_geometric_means'],
                    f'adjoint_over_uniform_{t}': d['adjoint / uniform_pass']['ratio_of_geometric_means'],
                    f'adjoint_free_over_uniform_{t}': d['adjoint_free / uniform_pass']['ratio_of_geometric_means'],
                    f'cheap_adjoint_over_cheap_{t}': d['cheap_adjoint / cheap']['ratio_of_geometric_means'],
                    f'retained_headroom_{t}': HEADROOM[cell][t]})
    CELL_ROWS.append(row)
CELLS = pd.DataFrame(CELL_ROWS)
CELLS.to_csv(PATHS['figures'] / 'phase_table.csv', index=False)
print(CELLS[['cell', 'm', 'hidden', 'R1', 'R3', 'candidate_regime_for_d4_1', 'R1_semilog', 'R1_staircase', 'amortised_price_ratio_to_step_doubling']].to_string(index=False))

import matplotlib.pyplot as plt

arms = [('amortised', 'C0'), ('cheap', 'C2'), ('cheap_adjoint', 'C3'), ('adjoint', 'C1'), ('adjoint_free', 'C4')]
fig, axes = plt.subplots(1, len(TARGETS), figsize=(4.6 * len(TARGETS), 4.2), sharey=True)
for ax, target in zip(np.atleast_1d(axes), TARGETS):
    for k, (arm, colour) in enumerate(arms):
        ax.scatter(CELLS[f'{arm}_over_uniform_{target}'], np.arange(len(CELLS)) + 0.12 * (k - 2), color=colour, label=arm, s=28)
    ax.axvline(1.0, color='grey', lw=1)
    ax.set_xscale('log'); ax.set_yticks(np.arange(len(CELLS))); ax.set_yticklabels(CELLS['cell'])
    ax.set_title(f'target: {target[3:]} of the gap left')
    ax.set_xlabel('compute vs uniform_pass (below 1: adaptive is cheaper)')
np.atleast_1d(axes)[0].legend(fontsize=8)
fig.suptitle(f'{RUN_ID}: validation, geometric-mean compute ratio at the real price (adjoint_free is hypothetical)', fontsize=10)
fig.savefig(PATHS['figures'] / 'compute_ratios.png', dpi=150, bbox_inches='tight')
plt.show()
plt.close(fig)

REPORT = {
    'run_id': RUN_ID, 'milestone': 'D4-2 (exploratory design study)', 'config_hash': CONFIG_HASH, 'frozen_sha256': FROZEN['sha256'], 'repo_commit': REPO_COMMIT,
    'correctness': CORRECTNESS, 'selected_theta': FROZEN['selected_theta'], 'selected_hidden': SELECTED_HIDDEN,
    'theta_selected_at_grid_edge': {f'{r["cell"]}/{r["arm"]}': bool(r['selected_at_grid_edge']) for r in TUNING_ROWS},
    'price_reports': PRICES, 'one_off_costs_and_break_even': BREAK_EVEN, 'retained_headroom': HEADROOM,
    'cells': CELLS.to_dict(orient='records'),
    'candidate_cells_for_d4_1': CELLS[CELLS['candidate_regime_for_d4_1']]['cell'].tolist(),
    'test_family_read': False,
    'claim_boundary': ('Exploratory design study on a train, a tuning and a validation family of the analytic D4 benchmark with a fixed goal per system; '
                       'the test family was not read. FLOPs are counted from the code path, not measured. adjoint_free is a hypothetical ceiling. '
                       'The amortised scorer is supervised by the adjoint-weighted error and is not a direct critic, so no H2 statement follows.'),
}
atomic_write_json(PATHS['reports'] / 'acceptance_report.json', REPORT)
lines = [f'# D4-2 `{RUN_ID}` (exploratory)', '', f"Correctness: **{'PASS' if CORRECTNESS['passed'] else 'FAIL'}** · D4-1 candidate cells: "
         f"**{len(REPORT['candidate_cells_for_d4_1'])}** of {len(CELLS)}", '',
         '| Cell | m | Hidden | R1 amortised beats uniform | R3 co-state weight helps cheap | Candidate | Amortised price / step-doubling |', '|---|---:|---:|---|---|---|---:|']
for _, r in CELLS.iterrows():
    lines.append(f"| {r['cell']} | {r['m']} | {r['hidden']} | {r['R1']} | {r['R3']} | {r['candidate_regime_for_d4_1']} | {r['amortised_price_ratio_to_step_doubling']:.3f} |")
lines += ['', REPORT['claim_boundary']]
atomic_write_text(PATHS['reports'] / 'run_summary.md', '\n'.join(lines) + '\n')
atomic_write_text(RUN_DIR / 'COMPLETE', datetime.datetime.now(datetime.UTC).isoformat() + '\n')
print('\n'.join(lines))
print('\nRun directory:', RUN_DIR)

## Reading the result

| Outcome in a cell | Meaning | Next step |
|---|---|---|
| R1 and R3 both hold | An amortised scorer pays at its measured price and the co-state weight helps a cheap estimator | A candidate regime for D4-1 (learned direct critic vs co-state-featured critic, goal varying per instance), gated separately, test family read once |
| R1 holds, R3 does not | Cheap scoring pays but the co-state weight adds nothing there | The co-state is not the source of the win; D4-1 has no headroom in that cell |
| R3 holds, R1 does not | The co-state weight helps a hand-built estimator, but the learned scorer does not pay | A non-learned cheap adjoint is the practical option; a learned scorer needs to be cheaper or better |
| Neither | No cheap scoring scheme tested pays at its price in that cell | Record it as a D4 negative result for that cell |

Read the price ratio column first: it shows where in `m` the learned scorer becomes cheaper than solving. Read the retained headroom, the one-off costs and the break-even instance counts before any statement about practicality. The sensitivity columns (`semilog`, `staircase`) say whether the primary reading depends on how a partial pass is priced.